# 📘 03 - Looping Over Files

So far, all your data has been typed into the notebook by hand. Real data lives in **files**. In this notebook you'll read a file line by line, turn each line into a dictionary, and send the result into the same kind of pipeline you built in Module 03b.

## ✅ Learning Goals
- Open a file safely with `with open(...) as f:`
- Loop over a file's lines, cleaning each one with `.strip()` and `.split()`
- Skip messy lines with `continue`
- Turn a file into a **list of dictionaries**, and wrap that in a function
- Handle a missing file with `try` / `except`

## ✍️ First, Create a Data File

Run this cell once. It writes `grades.txt` next to this notebook. It's deliberately a bit messy, because real files always are.

In [ ]:
lines = [
    "name,grade",
    "Alice,91",
    "Bob,84",
    "# Carol took the makeup exam",
    "Carol,95",
    "",
    "Dev,62",
    "Eli,78",
]

with open("grades.txt", "w") as f:
    for line in lines:
        f.write(line + "\n")

print("grades.txt created!")

## 📖 Reading a File, Line by Line

A file works like a list of lines, so a `for` loop walks through it one line at a time:

In [ ]:
with open("grades.txt") as f:
    for line in f:
        print(line)

🔮 **Why is there a blank line between every line?**

Each line in the file ends with an invisible newline character (`"\n"`), and `print()` adds another one. `.strip()` removes whitespace, including that newline, from both ends:

In [ ]:
with open("grades.txt") as f:
    for line in f:
        print(repr(line), "→", repr(line.strip()))

### What `with` Does

```python
with open("grades.txt") as f:
    ...           # the file is open inside this block
# ← the file is closed automatically here, even if an error happened
```

Without `with`, you'd have to remember to call `f.close()` yourself. With `with`, it happens automatically.

---

## ✂️ Turning a Line Into Data

🔮 Predict each value:

```python
line = "Alice,91\n"
line.strip()
line.strip().split(",")
line.strip().split(",")[1]
int(line.strip().split(",")[1])
```

(This is the "take it apart one step at a time" skill from Module 03b.)

✍️ **My prediction:** _(write it here BEFORE you run the next cell)_

In [ ]:
line = "Alice,91\n"
print(repr(line.strip()))
print(line.strip().split(","))
print(repr(line.strip().split(",")[1]))
print(int(line.strip().split(",")[1]))

## 🧹 Skipping the Mess With `continue`

The file has a header row, a comment, and a blank line. None of those are data. Use `continue` (from Notebook 02) to skip them:

In [ ]:
students = []

with open("grades.txt") as f:
    for line in f:
        line = line.strip()
        if line == "" or line.startswith("#") or line == "name,grade":
            continue                      # not data, so skip it
        name, grade = line.split(",")     # tuple unpacking!
        students.append({"name": name, "grade": int(grade)})

print(students)

That's a **list of dictionaries**, the same shape as `students` in Module 03b's pipeline notebook. The data now comes from a file instead of being typed in.

## 🧰 Wrap It in a Function

Reading the file is one step, so it should be one function:

In [ ]:
def read_grades(filename):
    """
    IN:   a filename (string)
    OUT:  a list of {"name": str, "grade": int} dictionaries
    DOES: reads the file, skipping blank lines, comments, and the header
    """
    students = []
    with open(filename) as f:
        for line in f:
            line = line.strip()
            if line == "" or line.startswith("#") or line == "name,grade":
                continue
            name, grade = line.split(",")
            students.append({"name": name, "grade": int(grade)})
    return students


def average(numbers):
    return sum(numbers) / len(numbers)

# The pipeline now starts at a file:
rows = read_grades("grades.txt")
grades = []
for r in rows:
    grades.append(r["grade"])
print("class average:", average(grades))

```text
 "grades.txt" ──► read_grades ──► list of dicts ──► (get grades) ──► average ──► 82.0
   (a file)                       (in memory)        (list of int)            (number)
```

When you get to Pandas, `pd.read_csv("grades.txt")` does the job of `read_grades` in one line. Now you know what it's doing behind the scenes.

---

## 🚦 When the File Isn't There: `try` / `except`

🔮 What happens when you try to open a file that doesn't exist?

In [ ]:
# ⚠️ This cell raises an error on purpose. Read the error message!
read_grades("no_such_file.txt")

The error's name is `FileNotFoundError`. You can **catch** it and handle it yourself instead of crashing:

```text
try:
    <code that might fail>
except SomeError:
    <what to do if it does>
```

In [ ]:
try:
    rows = read_grades("no_such_file.txt")
except FileNotFoundError:
    print("Couldn't find that file. Check the name and the folder.")
    rows = []

print("rows:", rows)

Only catch errors you **expect** and know how to handle. A missing file is a reasonable thing to catch. Wrapping all your code in `try` just hides bugs.

### 🔀 Cross-Language Note: `with` vs. C++ RAII

In C++, an `ifstream` closes itself when it goes out of scope (RAII). In Python, `with` gives you the same guarantee: the file closes when the block ends, even if an error happens inside it.

---

## 🏋️ Practice

### Practice 1 — Count the Lines (easy)
Loop over `grades.txt` and count **every** line, including the messy ones. Expected: `8`

In [ ]:
# Your code here

### Practice 2 — Only the Comments (medium)
Loop over `grades.txt` and print only the lines that start with `#` (without the trailing newline).

In [ ]:
# Your code here

### Practice 3 — A Safe Loader (harder)
Write `safe_read_grades(filename)` that calls `read_grades`, but returns an **empty list** (and prints a friendly message) if the file doesn't exist.

- `len(safe_read_grades("grades.txt"))` → `5`
- `safe_read_grades("nope.txt")` → `[]`

In [ ]:
def safe_read_grades(filename):
    # Your code here
    pass

print(len(safe_read_grades("grades.txt")))
print(safe_read_grades("nope.txt"))

## ✏️ Your Turn — File In, File Out

Build a pipeline that:

1. reads `grades.txt` with `read_grades`,
2. keeps the students whose grade is **above the class average**,
3. **writes** their names to a new file `honor_roll.txt`, one per line (use `open(..., "w")` and `f.write(name + "\n")`),
4. reads `honor_roll.txt` back and prints it to confirm.

Expected names: Alice, Bob, Carol.

In [ ]:
# Your pipeline here

## 🔥 Challenge (Optional) — A Bad Number

Add the line `"Fay,absent"` to the file (rewrite `grades.txt` with it included). Now `int("absent")` crashes `read_grades` with a `ValueError`.

Change `read_grades` so it **skips** any line whose grade isn't a number. Use a `try`/`except ValueError` **inside** the loop, together with `continue`.

In [ ]:
# Your code here